In [ ]:
!pip install sentence-transformers faiss-cpu openai

In [ ]:
import json
import logging
import pickle
from pathlib import Path
from typing import Optional

import faiss
import numpy as np
from openai import OpenAI
from sentence_transformers import SentenceTransformer

from wiki_ingestion import WikipediaIngester

In [ ]:
EMBED_MODEL       = "all-MiniLM-L6-v2"
CHAT_MODEL        = "llama3.1"
OLLAMA_BASE       = "http://localhost:11434/v1"
TOP_K             = 5
FAISS_INDEX_PATH  = Path("data/faiss_index/f1.index")
FAISS_META_PATH   = Path("data/faiss_index/f1_meta.pkl")

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
ingester = WikipediaIngester()
chunks   = ingester.load_all_chunks()

print(f"Total chunks loaded: {len(chunks)}")
print("\nSample chunk:")
print(json.dumps(chunks[0], indent=2))

In [ ]:
class F1VectorStore:
    """Local vector store: SentenceTransformer embeddings + FAISS index."""

    def __init__(
        self,
        embed_model: str  = EMBED_MODEL,
        index_path:  Path = FAISS_INDEX_PATH,
        meta_path:   Path = FAISS_META_PATH,
    ):
        self._index_path = Path(index_path)
        self._meta_path  = Path(meta_path)

        logger.info(f"Loading embedding model '{embed_model}' …")
        self._embedder = SentenceTransformer(embed_model)

        self._index = None
        self._meta  = []

        if self._index_path.exists() and self._meta_path.exists():
            logger.info("Loading FAISS index from disk …")
            self._index = faiss.read_index(str(self._index_path))
            with open(self._meta_path, "rb") as f:
                self._meta = pickle.load(f)
            logger.info(f"  Loaded {self._index.ntotal} vectors.")

    @property
    def count(self) -> int:
        return self._index.ntotal if self._index else 0

    def is_populated(self) -> bool:
        return self.count > 0

    def ingest_chunks(self, chunks: list[dict]) -> None:
        """Embed all chunks and build a FAISS flat index."""
        logger.info(f"Embedding {len(chunks)} chunks …")
        texts = [c["combined_text"] for c in chunks]
        embeddings = self._embedder.encode(
            texts,
            batch_size=64,
            show_progress_bar=True,
            convert_to_numpy=True,
        ).astype("float32")

        faiss.normalize_L2(embeddings)          # cosine via inner product
        dim   = embeddings.shape[1]
        index = faiss.IndexFlatIP(dim)
        index.add(embeddings)

        self._index = index
        self._meta  = chunks

        self._index_path.parent.mkdir(parents=True, exist_ok=True)
        faiss.write_index(index, str(self._index_path))
        with open(self._meta_path, "wb") as f:
            pickle.dump(chunks, f)
        logger.info(f"FAISS index saved ({index.ntotal} vectors).")

    def query(self, question: str, top_k: int = TOP_K) -> list[dict]:
        """Return top-k chunks most similar to the question."""
        q_vec = self._embedder.encode([question], convert_to_numpy=True).astype("float32")
        faiss.normalize_L2(q_vec)
        scores, indices = self._index.search(q_vec, top_k)

        passages = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue
            chunk = self._meta[idx]
            passages.append({
                "text":          chunk["text"],
                "combined_text": chunk["combined_text"],
                "article_title": chunk["article_title"],
                "section":       chunk["section"],
                "url":           chunk["url"],
                "score":         round(float(score), 4),
            })
        return passages

In [ ]:
store = F1VectorStore()

if store.is_populated():
    print(f"FAISS index already has {store.count} vectors — skipping ingestion.")
else:
    print(f"Ingesting {len(chunks)} chunks …")
    store.ingest_chunks(chunks)
    print(f"Done. Index now has {store.count} vectors.")

In [ ]:
test_question = "Who won the 2024 Formula One World Championship?"

passages = store.query(test_question, top_k=TOP_K)

print(f"Top {TOP_K} passages for: '{test_question}'\n")
for i, p in enumerate(passages, 1):
    print(f"[{i}] {p['article_title']} › {p['section']}  (score={p['score']})")
    print(f"    {p['text'][:200]}…\n")

In [ ]:
SYSTEM_PROMPT = """You are an expert Formula 1 analyst. Answer the user's question
using ONLY the provided context passages. If the context does not contain enough
information to answer confidently, say so clearly. Cite the article title when
you reference specific facts. Be concise and precise."""


class F1RAGPipeline:
    """Full retrieval-augmented generation pipeline (fully local)."""

    def __init__(self, vector_store: F1VectorStore, top_k: int = TOP_K):
        self._store  = vector_store
        self._top_k  = top_k
        # Ollama exposes an OpenAI-compatible API — no real key needed
        self._client = OpenAI(api_key="ollama", base_url=OLLAMA_BASE)

    def ask(self, question: str) -> dict:
        passages      = self._store.query(question, self._top_k)
        context_block = self._build_context(passages)
        answer        = self._generate(question, context_block)

        return {
            "question": question,
            "answer":   answer,
            "contexts": [p["text"] for p in passages],
            "sources":  [
                {
                    "title":   p["article_title"],
                    "section": p["section"],
                    "url":     p["url"],
                    "score":   p["score"],
                }
                for p in passages
            ],
        }

    def _build_context(self, passages: list[dict]) -> str:
        parts = []
        for i, p in enumerate(passages, 1):
            header = f"[{i}] {p['article_title']}"
            if p["section"]:
                header += f" › {p['section']}"
            parts.append(f"{header}\n{p['text']}")
        return "\n\n---\n\n".join(parts)

    def _generate(self, question: str, context: str) -> str:
        response = self._client.chat.completions.create(
            model=CHAT_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {question}"},
            ],
            temperature=0.2,
        )
        return response.choices[0].message.content.strip()

In [ ]:
pipeline = F1RAGPipeline(store)

questions = [
    "Who won the 2024 Formula One World Championship?",
    "What is the DRS system in Formula One?",
    "Which team did Ayrton Senna drive for?",
]

for q in questions:
    result = pipeline.ask(q)
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    print("Sources:")
    for s in result["sources"]:
        label = f"{s['title']} › {s['section']}" if s["section"] else s["title"]
        print(f"  • {label}  (score={s['score']})")
    print()

In [ ]:
q = "How many world championships did Lewis Hamilton win?"
passages = store.query(q, top_k=3)

print(f"Question: {q}\n")
for i, p in enumerate(passages, 1):
    print(f"--- Passage {i} (score={p['score']}) ---")
    print(f"Source: {p['article_title']} › {p['section']}")
    print(p["text"])
    print()